# Debug Scraper Antara Jatim

Local HTML first, then optional live list/article checks. Output list CSV: `csv/antara_jatim_list_debug.csv`.

In [1]:
from pathlib import Path
import sys
import importlib
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scrapers.common import cutoff_date, parse_date, records_to_df
import scrapers.antarajatim as scraper
scraper = importlib.reload(scraper)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 160)

SAMPLE_PATH = PROJECT_ROOT / 'html_samples/antarajatim.txt'
OUTPUT_PATH = PROJECT_ROOT / 'csv/antara_jatim_list_debug.csv'
MAX_PAGES = 200
TITLE_LIMIT = 90
cutoff = cutoff_date()

print('project:', PROJECT_ROOT)
print('module:', scraper.__file__)
print('sample:', SAMPLE_PATH)
print('cutoff:', cutoff.date())


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
module: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/antarajatim.py
sample: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/html_samples/antarajatim.txt
cutoff: 2026-02-28


In [2]:

def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df
    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(rows=('url', 'count'), newest=('published_dt', 'max'), oldest=('published_dt', 'min'))
        .reset_index()
        .tail(80)
    )
    return df


In [3]:

def parse_local_list(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')
    rows = []
    for card in soup.select('.card__post'):
        title_tag = card.select_one('.card__post__title h2 a[href]')
        date_tag = card.select_one('.card__post__author-info')
        image_tag = card.select_one('img')
        if not title_tag:
            continue
        rows.append({
            'page_num': 1,
            'list_page_url': scraper.BASE_URL,
            'title': title_tag.get_text(' ', strip=True),
            'url': urljoin(scraper.BASE_URL, title_tag['href']),
            'published_date': date_tag.get_text(' ', strip=True) if date_tag else None,
            'image_url': (image_tag.get('data-src') or image_tag.get('src')) if image_tag else None,
        })
    return records_to_df(rows)


## Local HTML list parse

In [4]:
html_text = SAMPLE_PATH.read_text(errors='replace')
local_df = parse_local_list(html_text)
local_df = ensure_debug_columns(local_df)
print_debug_rows(local_df)
local_df = audit_list(local_df)


page=001 | date=26 Juni 2026 23:03 | title=Ribuan umat Muslim di Malang doakan Prabowo dikuatkan pimpin bangsa
page=001 | date=21 jam lalu | title=Liburan di kawasan tiga pantai di Malang
page=001 | date=27 Juni 2026 20:39 | title=PDI Perjuangan Kota Malang dekatkan diri dengan anak muda lewat kompetisi e-sport
page=001 | date=27 Juni 2026 18:51 | title=Menkop siapkan koperasi pesantren jadi pemasok Koperasi Merah Putih
page=001 | date=27 Juni 2026 15:45 | title=Pemkab Malang komunikasikan realisasi hilirisasi garam dengan pusat
page=001 | date=26 Juni 2026 16:47 | title=Peneliti UB temukan empat spesies kumbang baru
page=001 | date=26 Juni 2026 09:48 | title=Pemkot Malang: Sejak 2025 tak ada kasus pemasungan disabilitas mental
page=001 | date=25 Juni 2026 20:34 | title=DPRD Malang dorong SiLPA Rp303 Miliar untuk pembiayaan UHC
page=001 | date=25 Juni 2026 16:50 | title=Polres Malang bangun sumur bor di tujuh desa antisipasi dampak kemarau
page=001 | date=25 Juni 2026 16:10 | title=Din

,month,count
0,2026-06,10



rows per page:


,page_num,rows,newest,oldest
0,1,10,2026-06-28 15:15:17.316386,2026-06-25 16:10:00


## Live list scrape

In [5]:
live_df = scraper.scrape_list(cutoff, max_pages=MAX_PAGES)
live_df = ensure_debug_columns(live_df)
print_debug_rows(live_df)
live_df = audit_list(live_df)


Scraping Antara Jatim page 1: https://jatim.antaranews.com/kabar-jatim/malang-raya
Scraping Antara Jatim page 2: https://jatim.antaranews.com/kabar-jatim/malang-raya/2
Scraping Antara Jatim page 3: https://jatim.antaranews.com/kabar-jatim/malang-raya/3
Scraping Antara Jatim page 4: https://jatim.antaranews.com/kabar-jatim/malang-raya/4
Scraping Antara Jatim page 5: https://jatim.antaranews.com/kabar-jatim/malang-raya/5
Scraping Antara Jatim page 6: https://jatim.antaranews.com/kabar-jatim/malang-raya/6
Scraping Antara Jatim page 7: https://jatim.antaranews.com/kabar-jatim/malang-raya/7
Scraping Antara Jatim page 8: https://jatim.antaranews.com/kabar-jatim/malang-raya/8
Scraping Antara Jatim page 9: https://jatim.antaranews.com/kabar-jatim/malang-raya/9
Scraping Antara Jatim page 10: https://jatim.antaranews.com/kabar-jatim/malang-raya/10
Scraping Antara Jatim page 11: https://jatim.antaranews.com/kabar-jatim/malang-raya/11
Scraping Antara Jatim page 12: https://jatim.antaranews.com/kab

,month,count
0,2026-02,4
1,2026-03,112
2,2026-04,96
3,2026-05,85
4,2026-06,69



rows per page:


,page_num,rows,newest,oldest
0,1,10,2026-06-28 14:16:17.805427,2026-06-25 16:10:00
1,2,10,2026-06-25 13:47:00.000000,2026-06-20 15:08:00
2,3,10,2026-06-20 12:17:00.000000,2026-06-17 21:33:00
3,4,10,2026-06-17 17:42:00.000000,2026-06-13 16:14:00
4,5,10,2026-06-13 11:59:00.000000,2026-06-09 20:01:00
5,6,10,2026-06-09 18:35:00.000000,2026-06-04 16:00:00
6,7,10,2026-06-03 20:17:00.000000,2026-05-31 23:00:00
7,8,10,2026-05-31 21:22:00.000000,2026-05-28 19:10:00
8,9,10,2026-05-28 14:01:00.000000,2026-05-25 21:20:00
9,10,10,2026-05-25 17:25:00.000000,2026-05-20 20:17:00


## Save debug list CSV

In [6]:
df_to_save = live_df if 'live_df' in globals() and len(live_df) else local_df
OUTPUT_PATH.parent.mkdir(exist_ok=True)
df_to_save.to_csv(OUTPUT_PATH, index=False)
print('saved:', OUTPUT_PATH)


saved: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/antara_jatim_list_debug.csv


## Optional article sample check

In [7]:
sample_rows = (live_df if 'live_df' in globals() and len(live_df) else local_df).head(5)
article_rows = []
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")
article_df = pd.DataFrame(article_rows)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)


[1] content_len=0 | Liburan di kawasan tiga pantai di Malang
[2] content_len=2147 | PDI Perjuangan Kota Malang dekatkan diri dengan anak muda lewat kompetisi e-sport
[3] content_len=2502 | Menkop siapkan koperasi pesantren jadi pemasok Koperasi Merah Putih
[4] content_len=2247 | Pemkab Malang komunikasikan realisasi hilirisasi garam dengan pusat
[5] content_len=2221 | Ribuan umat Muslim di Malang doakan Prabowo dikuatkan pimpin bangsa


,title,published_date,content,url
0,Liburan di kawasan tiga pantai di Malang,2026-06-28,NaN,https://jatim.antaranews.com/foto/1077469/liburan-di-kawasan-tiga-pantai-di-malang
1,PDI Perjuangan Kota Malang dekatkan diri dengan anak muda lewat kompetisi e-sport,"Pewarta: Ananto Pradana Editor : Astrid Faidlatul Habibah COPYRIGHT © ANTARA 2026 Dilarang keras mengambil konten, melakukan crawling atau pengindeksan otom...","Partai harus mengikuti dinamika di dalam tatanan masyarakat, khususnya ke anak-anak zaman sekarang\nMalang Raya (ANTARA) - Dewan Pimpinan Cabang (DPC) PDI P...",https://jatim.antaranews.com/berita/1077367/pdi-perjuangan-kota-malang-dekatkan-diri-dengan-anak-muda-lewat-kompetisi-e-sport
2,Menkop siapkan koperasi pesantren jadi pemasok Koperasi Merah Putih,"Pewarta: Ananto Pradana Editor : Abdul Hakim COPYRIGHT © ANTARA 2026 Dilarang keras mengambil konten, melakukan crawling atau pengindeksan otomatis untuk AI...","Malang, Jawa Timur (ANTARA) - Menteri Koperasi (Menkop) Ferry Juliantono berkomitmen mengawal perkembangan koperasi pondok pesantren (ponpes) di bawah Majel...",https://jatim.antaranews.com/berita/1077335/menkop-siapkan-koperasi-pesantren-jadi-pemasok-koperasi-merah-putih
3,Pemkab Malang komunikasikan realisasi hilirisasi garam dengan pusat,"Pewarta: Ananto Pradana Editor : Vicki Febrianto COPYRIGHT © ANTARA 2026 Dilarang keras mengambil konten, melakukan crawling atau pengindeksan otomatis untu...","Malang, Jawa Timur (ANTARA) - Pemerintah Kabupaten (Pemkab) Malang, Jawa Timur menyatakan realisasi hilirisasi garam di wilayah pesisir Malang masih dikomun...",https://jatim.antaranews.com/berita/1077292/pemkab-malang-komunikasikan-realisasi-hilirisasi-garam-dengan-pusat
4,Ribuan umat Muslim di Malang doakan Prabowo dikuatkan pimpin bangsa,"Pewarta: Ananto Pradana Editor : Vicki Febrianto COPYRIGHT © ANTARA 2026 Dilarang keras mengambil konten, melakukan crawling atau pengindeksan otomatis untu...","Malang Raya (ANTARA) - Ribuan umat Muslim peserta acara Istighatsah dan Doa Kebangsaan yang diselenggarakan di Pondok Pesantren An Nur 2 Al-Murtadlo, Kabupa...",https://jatim.antaranews.com/berita/1077175/ribuan-umat-muslim-di-malang-doakan-prabowo-dikuatkan-pimpin-bangsa
